In [1]:
import sys
import os
sys.path.append(os.pardir)

In [2]:
import pandas as pd

In [3]:
from datetime import datetime, timedelta

In [4]:
from config import GRIDSTATUS_API_KEY

In [5]:
from gridstatusio import GridStatusClient

# Load Data From Gridstatus API

## Basic test example from the docs
- free plan has a 1 million rows per month limit
- add a 'limit' to all get_dataset request to avoid blowing through the limit

In [7]:
#Test
client = GridStatusClient(api_key = GRIDSTATUS_API_KEY)

In [7]:
#returns a pandas df
data = client.get_dataset('ercot_fuel_mix', limit=100, start='2025-01-01', end='2025-01-02')

2026-06-30 22:48:30 - INFO - Fetching Page 1...
2026-06-30 22:48:30 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_fuel_mix/query
2026-06-30 22:48:30 - INFO - Params: {'start_time': Timestamp('2025-01-01 00:00:00'), 'end_time': Timestamp('2025-01-02 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 100, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-06-30 22:48:31 - INFO - Done in 1.1 seconds. 
2026-06-30 22:48:31 - INFO - Total rows: 100/100 (100.0% of limit)
2026-06-30 22:48:31 - INFO - Total number of rows: 100


In [8]:
data.head()

,interval_start_utc,interval_end_utc,coal_and_lignite,hydro,nuclear,power_storage,solar,wind,natural_gas,other
0,2025-01-01 00:00:00+00:00,2025-01-01 00:05:00+00:00,9777.7,224.4,5089.8,3456.3,0.4,4392.2,25303.8,0.0
1,2025-01-01 00:05:00+00:00,2025-01-01 00:10:00+00:00,9778.6,224.5,5091.5,3288.6,0.4,4528.6,25249.4,0.0
2,2025-01-01 00:10:00+00:00,2025-01-01 00:15:00+00:00,9787.5,223.8,5092.6,3204.9,0.4,4668.8,25178.9,0.0
3,2025-01-01 00:15:00+00:00,2025-01-01 00:20:00+00:00,9834.4,223.4,5089.3,2910.9,0.5,4782.5,25268.9,0.0
4,2025-01-01 00:20:00+00:00,2025-01-01 00:25:00+00:00,9843.1,223.6,5090.1,2833.7,0.4,4915.6,25254.0,0.0


## Checking the API usage

In [23]:
usage = client.get_api_usage()

In [24]:
usage

{'plan_name': 'Free',
 'limits': {'api_rows_returned_limit': 500000,
  'api_requests_limit': 250,
  'api_rows_per_response_limit': 50000,
  'per_second_api_rate_limit': 1,
  'per_minute_api_rate_limit': 30,
  'per_hour_api_rate_limit': 600},
 'current_usage_period_start': '2026-07-01T00:00:00Z',
 'current_usage_period_end': '2026-08-01T00:00:00Z',
 'current_period_usage': {'total_requests': 6, 'total_api_rows_returned': 472}}

### Retry logic for heavy pagination
- client will retry from rate limit error 429 and server errors 5XX

In [11]:
client = GridStatusClient(
    max_retries=3,        # Maximum retries (default: 5)
    base_delay=1.0,       # Base delay in seconds (default: 2.0)
    exponential_base=1.5, # Exponential backoff multiplier (default: 2.0)
)

Exception: No API key provided. Either pass an api_key to the                 GridStatusClient constructor or set the                 GRIDSTATUS_API_KEY environment variable.

In [16]:
### Pulling spp load
#2024-03-18 06:00:00 	2024-03-22 05:00:00
spp_data = client.get_dataset('spp_load_hourly', limit=100, start='2024-03-19', end='2024-03-22')

2026-06-30 22:56:34 - INFO - Fetching Page 1...
2026-06-30 22:56:34 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-06-30 22:56:34 - INFO - Params: {'start_time': Timestamp('2024-03-19 00:00:00'), 'end_time': Timestamp('2024-03-22 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 100, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-06-30 22:56:35 - INFO - Done in 0.37 seconds. 
2026-06-30 22:56:35 - INFO - Total rows: 100/100 (100.0% of limit)
2026-06-30 22:56:35 - INFO - Total number of rows: 100


In [ ]:
spp_data.tail(50)

### Filtering the data
- balancing_area_name has SPP, SWPW and another?
-  The Areas in this dataset are the legacy balancing authorities for the region that SPP serves
-  so filter the "balancing_are_name" to just "SPP"
-  what is the 'control_zone_name'? I think we need to sum across all the control zones to get the total system SPP region load
-  timestamp is "interval_start_utc"

In [20]:
spp_data = client.get_dataset(
    'spp_load_hourly',  
    start='2024-03-18', 
    end='2024-03-20',
    columns = ["interval_start_utc", "balancing_area_name", "control_zone_name", "forecast_area_type", "load"],
    filter_column = "control_zone_name",
    filter_value = "SYSTEM_TOTAL",
    limit=100
)

2026-06-30 23:03:39 - INFO - Fetching Page 1...
2026-06-30 23:03:39 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-06-30 23:03:39 - INFO - Params: {'start_time': Timestamp('2024-03-18 00:00:00'), 'end_time': Timestamp('2024-03-20 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 100, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': 'control_zone_name', 'filter_value': 'SYSTEM_TOTAL', 'filter_operator': '=', 'columns': 'interval_start_utc,balancing_area_name,control_zone_name,forecast_area_type,load', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-06-30 23:03:39 - INFO - Done in 0.47 seconds. 
2026-06-30 23:03:39 - INFO - Total rows: 48/100 (48.0% of limit)
2026-06-30 23:03:39 - INFO - Total number of rows: 48


In [ ]:
spp_data.head(40)

In [ ]:
spp_data.balancing_area_name.unique()

In [ ]:
spp_data.forecast_area_type.unique()

In [25]:
END_DATE = datetime(2025, 12, 31)

In [39]:
file_name = f"rawgs_{END_DATE.isoformat()[:-9]}.csv"
file_name

'rawgs_2025-12-31.csv'

In [44]:
def fetch_load(client, start: datetime, end: datetime) -> pd.DataFrame:

    df = client.get_dataset(
        "spp_load_hourly",
        start = start.isoformat(),
        end = end.isoformat(),
        columns = ["interval_start_utc", "balancing_area_name", "control_zone_name", "forecast_area_type", "load"],
        filter_column = "control_zone_name",
        filter_value = "SYSTEM_TOTAL",
    )

    file_name = f"rawgs_{END_DATE.isoformat()[:-9]}.csv"
    path = "../data/raw/"
    df.to_csv(os.path.join(path, file_name), index = True)

    return df

In [42]:
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 4)

In [45]:
test_df = fetch_load(client, start_date, end_date)

2026-06-30 23:35:10 - INFO - Fetching Page 1...
2026-06-30 23:35:10 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-06-30 23:35:10 - INFO - Params: {'start_time': Timestamp('2024-01-01 00:00:00'), 'end_time': Timestamp('2024-01-04 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': 'control_zone_name', 'filter_value': 'SYSTEM_TOTAL', 'filter_operator': '=', 'columns': 'interval_start_utc,balancing_area_name,control_zone_name,forecast_area_type,load', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-06-30 23:35:10 - INFO - Done in 0.47 seconds. 
2026-06-30 23:35:10 - INFO - Total number of rows: 72


## Transform Load

In [ ]:
## Filter out for NC load
filter_data = spp_data[spp_data["forecast_area_type"] == "CF"]

In [ ]:
#Aggregate accros control zones
total_load = spp_data.groupby("interval_start_utc", as_index = False).agg(load = ("load", "sum"))

In [ ]:
total_load.head()

In [ ]:
total_load.dtypes

In [ ]:
total_load.index

In [ ]:
#set the datetime index and frequency
total_load.set_index("interval_start_utc", inplace = True)
total_load.index.freq = 'h'

In [ ]:
total_load.head()

In [ ]:
total_load.index

In [ ]:
#time zone is us, UTC
datetime64(precision, [timezone])
'2025-01-01 00:00:00+00:00'
'2025-01-01 00:00:00+00:00'

In [ ]:
filter_data = spp_data[spp_data["forecast_area_type"] == "CF"]
total_load = spp_data.groupby("interval_start_utc", as_index = False).agg(load = ("load", "sum"))
total_load.set_index("interval_start_utc", inplace = True)
total_load.index.freq = 'h'

In [ ]:
spp_data["forecast_area_type"] == "CF"

In [ ]:
df = spp_data.copy()
"""
(    
    df.groupby("interval_start_utc", as_index = False).agg(load = ("load", "sum"))
    .set_index("interval_start_utc", inplace = True)
    .index.freq = 'h'
)
"""

In [ ]:
#can you function chain with group by?
df.groupby("interval_start_utc", as_index = False).agg(load = ("load", "sum"))

In [ ]:
df.head()

# Pulling Historical Temp Forecast
- OKC: 35.481918, -97.508469
- KC: 39.099724, -94.578331
- Sioux Falls: 43.536388, -96.731667

In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 39.0997,
	"longitude": -94.5786,
	"start_date": "2025-01-01",
	"end_date": "2025-01-02",
	"hourly": "temperature_2m",
}

In [ ]:
responses = openmeteo.weather_api(url, params = params)

In [ ]:
response = responses[0]

In [ ]:
#print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")

In [ ]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

In [ ]:
hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

In [ ]:
#hourly_data

In [ ]:
hourly_data["temperature"] = hourly_temperature_2m

In [ ]:
hourly_df = pd.DataFrame(data = hourly_data)

In [ ]:
hourly_df.head()

In [ ]:
hourly_df.dtypes

In [ ]:
hourly_df.set_index("date", inplace = True)
hourly_df.index.freq = 'h'
hourly_df.head()

In [ ]:
hourly_df.index